# Amazon ML Challenge: Stage 3 — Business Entity Matcher & Classifier Pipeline

This notebook implements the end-to-end Machine Learning pipeline for resolving business entities across independent data sources (Source 1 reference vs. Source 2 & Source 3 targets):

1. **Config & Environment Setup**: Multi-path resolution, directory creation, seed initialization, and structured logging.
2. **Checkpoint 1 — Data Ingestion**: Robust loading of Source 1, Source 2, Source 3, and Ground Truth.
3. **Checkpoint 2 — Script-Preserving Normalization**: Unicode NFKC normalization and script identification (Devanagari, Telugu, Tamil, Kannada, etc.) without lossy transliteration.
4. **Checkpoint 3 — Token & Geography Extraction**: Extracting numeric tokens, postal codes (US 5-digit ZIPs / India 6-digit PINs), and word sets.
5. **Checkpoint 4 — Leakage-Free Train/Validation Split**: S1 entity-level split to prevent data contamination across splits.
6. **Checkpoint 5 — Multi-Rule Inverted Indexing**: Fast indexing of blocking keys with size limits to prevent combinatorial explosion.
7. **Checkpoint 6 — Hard Negative Mining & Feature Checkpoint**: Pairing true matches with hard negatives from the same blocking neighborhoods, saved to disk (`training_pairs.parquet`).
8. **Checkpoint 7 — CatBoost Training**: GPU-accelerated training (RTX 4050 `task_type='GPU'`) with automatic CPU fallback and model saving (`entity_matcher_catboost.cbm`).
9. **Checkpoint 8 — Validation Candidate Scoring**: Generating validation candidates and inferring match probabilities, saved to disk (`validation_predictions.parquet`).
10. **Checkpoint 9 — Threshold Search for Macro $F_{0.5}$**: Optimizing decision boundary for Macro $F_{0.5}$ with explicit singleton scoring.
11. **Checkpoint 10 — Detailed Diagnostics & Metrics Export**: Full metrics logging and JSON export (`validation_metrics.json`).
12. **Checkpoint 11 — Final Submission Generation**: Generating properly formatted `matching_results.tsv` and `candidate_pairs.tsv` conforming to official challenge rules.


## 1. Setup, Imports, Paths & Logging Configuration

We configure logging with exact timestamps, memory status, and path resolution that automatically discovers datasets either in `../data` or `../student_resource/dataset/train`.


In [1]:
import os
import sys
import re
import gc
import math
import json
import time
import random
import logging
import unicodedata
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# String similarity engine
from rapidfuzz.fuzz import (
    ratio,
    WRatio,
    token_sort_ratio,
    token_set_ratio
)

# Gradient Boosting
from catboost import CatBoostClassifier, Pool

# ------------------------------------------------------------
# LOGGING CONFIGURATION
# ------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger("EntityMatcher")

def log(msg):
    logger.info(msg)

# ------------------------------------------------------------
# REPRODUCIBILITY
# ------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ------------------------------------------------------------
# PATH RESOLUTION & DIRECTORY CREATION
# ------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

DATA_DIR = PROJECT_ROOT / "student_resource" / "dataset" / "train"
if not DATA_DIR.exists():
    DATA_DIR = PROJECT_ROOT / "data"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "model"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def find_file(preferred_dir, *names):
    for name in names:
        p = preferred_dir / name
        if p.exists():
            return p
    # Fallback to student_resource
    alt_dir = PROJECT_ROOT / "student_resource" / "dataset" / "train"
    for name in names:
        p = alt_dir / name
        if p.exists():
            return p
    return preferred_dir / names[0]

S1_PATH = find_file(DATA_DIR, "source1.tsv", "train_source1.tsv")
S2_PATH = find_file(DATA_DIR, "source2.tsv", "train_source2.tsv")
S3_PATH = find_file(DATA_DIR, "source3.tsv", "train_source3.tsv")
GT_PATH = find_file(DATA_DIR, "ground_truth.tsv", "train_ground_truth.tsv")

# Checkpoint file paths
MODEL_PATH = OUTPUT_DIR / "entity_matcher_catboost.cbm"
TRAIN_PAIRS_PATH = OUTPUT_DIR / "training_pairs.parquet"
VAL_PRED_PATH = OUTPUT_DIR / "validation_predictions.parquet"
FINAL_MATCHING_PATH = OUTPUT_DIR / "matching_results.tsv"
FINAL_CANDIDATES_PATH = OUTPUT_DIR / "candidate_pairs.tsv"
METRICS_PATH = OUTPUT_DIR / "validation_metrics.json"

log(f"Project Root: {PROJECT_ROOT.resolve()}")
log(f"Data Dir: {DATA_DIR.resolve()} (exists: {DATA_DIR.exists()})")
log(f"Output Checkpoints Dir: {OUTPUT_DIR.resolve()}")
log(f"S1 Path: {S1_PATH.name} ({'Found' if S1_PATH.exists() else 'Missing'})")
log(f"S2 Path: {S2_PATH.name} ({'Found' if S2_PATH.exists() else 'Missing'})")
log(f"S3 Path: {S3_PATH.name} ({'Found' if S3_PATH.exists() else 'Missing'})")
log(f"GT Path: {GT_PATH.name} ({'Found' if GT_PATH.exists() else 'Missing'})")


2026-09-26 16:37:35 [INFO] Project Root: D:\ml-model\amazon-ml
2026-09-26 16:37:35 [INFO] Data Dir: D:\ml-model\amazon-ml\student_resource\dataset\train (exists: True)
2026-09-26 16:37:35 [INFO] Output Checkpoints Dir: D:\ml-model\amazon-ml\outputs\model
2026-09-26 16:37:35 [INFO] S1 Path: train_source1.tsv (Found)
2026-09-26 16:37:35 [INFO] S2 Path: train_source2.tsv (Found)
2026-09-26 16:37:35 [INFO] S3 Path: train_source3.tsv (Found)
2026-09-26 16:37:35 [INFO] GT Path: train_ground_truth.tsv (Found)


## 2. Pipeline Hyperparameters & Checkpoint Settings

Configure sample sizes for fast development vs. full-scale execution, negative sampling ratio, blocking depth, and GPU settings.


In [2]:
# ------------------------------------------------------------
# HYPERPARAMETERS
# ------------------------------------------------------------
# Set SAMPLE_S1_SIZE to an integer (e.g., 20_000) for rapid iteration,
# or set to None to run on the entire training set.
SAMPLE_S1_SIZE = 25_000

# Validation split fraction (S1 entity level)
VAL_FRACTION = 0.10

# Ratio of hard negatives to positives per anchor
NEGATIVE_RATIO = 4

# Maximum candidates collected per blocking rule for an S1 entity
TOPK_PER_RULE = 30

# Maximum candidates retained after cheap ranking per source
FINAL_TOPK = 100

# CatBoost Configuration
CATBOOST_ITERATIONS = 1200
CATBOOST_DEPTH = 8
CATBOOST_LEARNING_RATE = 0.05

# Threshold search range for Macro F0.5
THRESHOLDS = np.arange(0.20, 0.96, 0.02)

# Checkpoint Flags: Set to True to reuse existing on-disk parquet/model files
USE_CACHED_TRAIN_PAIRS = TRAIN_PAIRS_PATH.exists()
USE_CACHED_MODEL = MODEL_PATH.exists()
USE_CACHED_VAL_PRED = VAL_PRED_PATH.exists()

log(f"Settings: SAMPLE_S1_SIZE={SAMPLE_S1_SIZE}, NEGATIVE_RATIO={NEGATIVE_RATIO}, FINAL_TOPK={FINAL_TOPK}")
log(f"Checkpoints found: Pairs={USE_CACHED_TRAIN_PAIRS}, Model={USE_CACHED_MODEL}, ValPred={USE_CACHED_VAL_PRED}")


2026-09-26 16:37:56 [INFO] Settings: SAMPLE_S1_SIZE=25000, NEGATIVE_RATIO=4, FINAL_TOPK=100
2026-09-26 16:37:56 [INFO] Checkpoints found: Pairs=True, Model=True, ValPred=True


## Checkpoint 1: Raw Data Ingestion & Schema Audit

Loading business entity records across the three sources. If `SAMPLE_S1_SIZE` is specified, we draw a representative sample that preserves true matches from Ground Truth to ensure zero-loss recall testing.


In [3]:
t0 = time.time()
log("Loading Raw TSV Data...")

# Read Ground Truth
gt_df = pd.read_csv(GT_PATH, sep="\t", dtype=str, keep_default_na=False)
log(f"Ground Truth loaded: {len(gt_df):,} rows")

if SAMPLE_S1_SIZE and SAMPLE_S1_SIZE < len(gt_df):
    log(f"Sampling {SAMPLE_S1_SIZE:,} S1 entities and anchoring target S2/S3 records...")
    # Sample S1 IDs from ground truth
    sampled_gt = gt_df.sample(n=SAMPLE_S1_SIZE, random_state=SEED).copy()
    s1_ids_needed = set(sampled_gt["source1_entity_id"])
    
    # Extract matched S2 and S3 IDs
    s2_match_ids = set()
    s3_match_ids = set()
    for ids_str in sampled_gt["matched_entity_ids"]:
        if ids_str:
            for mid in ids_str.split(","):
                mid = mid.strip()
                if mid.startswith("S2-"):
                    s2_match_ids.add(mid)
                elif mid.startswith("S3-"):
                    s3_match_ids.add(mid)
    
    # Read S1 filtered
    s1 = pd.read_csv(S1_PATH, sep="\t", dtype=str, keep_default_na=False)
    s1 = s1[s1["entity_id"].isin(s1_ids_needed)].reset_index(drop=True)
    
    # Read S2 and S3 (including true matches + realistic background distractors)
    s2_full = pd.read_csv(S2_PATH, sep="\t", dtype=str, keep_default_na=False)
    s2_matches = s2_full[s2_full["entity_id"].isin(s2_match_ids)]
    s2_distractors = s2_full[~s2_full["entity_id"].isin(s2_match_ids)].sample(
        n=min(len(s2_full) - len(s2_matches), SAMPLE_S1_SIZE * 2), random_state=SEED
    )
    s2 = pd.concat([s2_matches, s2_distractors], ignore_index=True).drop_duplicates(subset=["entity_id"]).reset_index(drop=True)
    del s2_full
    gc.collect()

    s3_full = pd.read_csv(S3_PATH, sep="\t", dtype=str, keep_default_na=False)
    s3_matches = s3_full[s3_full["entity_id"].isin(s3_match_ids)]
    s3_distractors = s3_full[~s3_full["entity_id"].isin(s3_match_ids)].sample(
        n=min(len(s3_full) - len(s3_matches), SAMPLE_S1_SIZE * 2), random_state=SEED
    )
    s3 = pd.concat([s3_matches, s3_distractors], ignore_index=True).drop_duplicates(subset=["entity_id"]).reset_index(drop=True)
    del s3_full
    gc.collect()

    gt = sampled_gt.reset_index(drop=True)
else:
    log("Loading full dataset across all files...")
    s1 = pd.read_csv(S1_PATH, sep="\t", dtype=str, keep_default_na=False)
    s2 = pd.read_csv(S2_PATH, sep="\t", dtype=str, keep_default_na=False)
    s3 = pd.read_csv(S3_PATH, sep="\t", dtype=str, keep_default_na=False)
    gt = gt_df

log(f"Data Loaded in {time.time() - t0:.2f}s:")
log(f"  S1: {len(s1):,} rows, {s1.shape[1]} columns")
log(f"  S2: {len(s2):,} rows, {s2.shape[1]} columns")
log(f"  S3: {len(s3):,} rows, {s3.shape[1]} columns")
log(f"  GT: {len(gt):,} records")

display(s1.head(3))


2026-09-26 16:37:58 [INFO] Loading Raw TSV Data...
2026-09-26 16:38:01 [INFO] Ground Truth loaded: 2,206,821 rows
2026-09-26 16:38:01 [INFO] Sampling 25,000 S1 entities and anchoring target S2/S3 records...
2026-09-26 16:38:36 [INFO] Data Loaded in 38.63s:
2026-09-26 16:38:36 [INFO]   S1: 25,000 rows, 4 columns
2026-09-26 16:38:36 [INFO]   S2: 92,060 rows, 4 columns
2026-09-26 16:38:36 [INFO]   S3: 94,674 rows, 4 columns
2026-09-26 16:38:36 [INFO]   GT: 25,000 records


,entity_id,business_name,business_address,country
0,S1-708796012,Board of Consumer Affairs,"122 Dayton Yellow Springs Road, Unit 4, Fairbo...",US
1,S1-483281264,Louisa's Group,"599 Mountain Shadow Lane, Maryville, TN",US
2,S1-387694500,Bombay Power Private Limited,"No: 32/2, 2Nd Floor, Mcc Towers, Opp To Sigma ...",India


## Checkpoint 2: Script-Preserving Normalization & Script Detection

### Why We Preserve Indic Scripts:
In multi-source Indian business entity resolution, converting Devanagari (`इंडो`), Kannada (`ಕರ್ನಾಟಕ`), Telugu, or Tamil into English using generic transliteration introduces phoneme distortions. Character-level and n-gram similarity works reliably within the native script. We normalize Unicode (NFKC), clean punctuation, and identify the primary script for each record.


In [4]:
t0 = time.time()
log("Applying Unicode Normalization and Script Detection...")

def normalize_unicode(text):
    if text is None:
        return ""
    text = str(text).strip()
    if not text:
        return ""
    text = unicodedata.normalize("NFKC", text).lower()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_name(text):
    return normalize_unicode(text)

def normalize_address(text):
    return normalize_unicode(text)

SCRIPT_PATTERNS = {
    "DEVANAGARI": r"[\u0900-\u097F]",
    "BENGALI": r"[\u0980-\u09FF]",
    "GURMUKHI": r"[\u0A00-\u0A7F]",
    "GUJARATI": r"[\u0A80-\u0AFF]",
    "ORIYA": r"[\u0B00-\u0B7F]",
    "TAMIL": r"[\u0B80-\u0BFF]",
    "TELUGU": r"[\u0C00-\u0C7F]",
    "KANNADA": r"[\u0C80-\u0CFF]",
    "MALAYALAM": r"[\u0D00-\u0D7F]",
}

def detect_script(text):
    if not text:
        return "EMPTY"
    counts = {}
    for script, pattern in SCRIPT_PATTERNS.items():
        cnt = len(re.findall(pattern, text))
        if cnt > 0:
            counts[script] = cnt
    latin_count = len(re.findall(r"[a-zA-Z]", text))
    if latin_count > 0:
        counts["LATIN"] = latin_count
    if not counts:
        return "OTHER"
    if len(counts) > 1:
        return "MIXED"
    return max(counts, key=counts.get)

# Process DataFrames
for name, df in [("S1", s1), ("S2", s2), ("S3", s3)]:
    df["name_norm"] = df["business_name"].map(normalize_name)
    df["address_norm"] = df["business_address"].map(normalize_address)
    df["name_script"] = df["business_name"].map(detect_script)
    df["address_script"] = df["business_address"].map(detect_script)
    log(f"Normalized {name}: Top scripts -> {dict(df['name_script'].value_counts().head(3))}")

log(f"Normalization & script tagging complete in {time.time() - t0:.2f}s")
display(s1[["business_name", "name_norm", "name_script", "country"]].head(5))


2026-09-26 16:38:58 [INFO] Applying Unicode Normalization and Script Detection...
2026-09-26 16:38:59 [INFO] Normalized S1: Top scripts -> {'LATIN': 25000}
2026-09-26 16:39:02 [INFO] Normalized S2: Top scripts -> {'LATIN': 83345, 'DEVANAGARI': 4712, 'KANNADA': 686}
2026-09-26 16:39:04 [INFO] Normalized S3: Top scripts -> {'LATIN': 89715, 'DEVANAGARI': 2392, 'MIXED': 604}
2026-09-26 16:39:04 [INFO] Normalization & script tagging complete in 5.77s


,business_name,name_norm,name_script,country
0,Board of Consumer Affairs,board of consumer affairs,LATIN,US
1,Louisa's Group,louisa s group,LATIN,US
2,Bombay Power Private Limited,bombay power private limited,LATIN,India
3,Data Materials Networks,data materials networks,LATIN,US
4,Technocast Chit Private Limited,technocast chit private limited,LATIN,India


## Checkpoint 3: Extraction of Numeric Tokens, Postal Codes & Word Sets

Numeric components (door numbers, plot numbers) and postal codes (US 5-digit ZIPs / India 6-digit PIN codes) are extracted for precise geographic filtering.


In [5]:
t0 = time.time()
log("Extracting numeric tokens, postal codes, and word token sets...")

def extract_numbers(text):
    if not text:
        return set()
    return set(re.findall(r"\d+", str(text)))

def extract_postal(text, country):
    nums = re.findall(r"\d+", str(text))
    if not nums:
        return ""
    country_upper = str(country).strip().upper()
    if country_upper in ("US", "USA"):
        for n in nums:
            if len(n) == 5:
                return n
    if country_upper in ("INDIA", "IN"):
        for n in nums:
            if len(n) == 6:
                return n
    return ""

def tokenize(text):
    if not text:
        return set()
    return set(x for x in text.split() if x)

for name, df in [("S1", s1), ("S2", s2), ("S3", s3)]:
    df["number_tokens"] = df["address_norm"].map(extract_numbers)
    df["postal"] = [extract_postal(a, c) for a, c in zip(df["address_norm"], df["country"])]
    df["name_tokens"] = df["name_norm"].map(tokenize)
    df["address_tokens"] = df["address_norm"].map(tokenize)
    postal_coverage = (df["postal"] != "").mean() * 100
    log(f"{name}: Postal code extracted for {postal_coverage:.1f}% of records")

log(f"Tokenization complete in {time.time() - t0:.2f}s")
display(s1[["entity_id", "postal", "number_tokens", "name_tokens"]].head(3))


2026-09-26 16:39:23 [INFO] Extracting numeric tokens, postal codes, and word token sets...
2026-09-26 16:39:24 [INFO] S1: Postal code extracted for 6.6% of records
2026-09-26 16:39:25 [INFO] S2: Postal code extracted for 6.3% of records
2026-09-26 16:39:26 [INFO] S3: Postal code extracted for 6.3% of records
2026-09-26 16:39:26 [INFO] Tokenization complete in 2.61s


,entity_id,postal,number_tokens,name_tokens
0,S1-708796012,,"{4, 122}","{board, consumer, affairs, of}"
1,S1-483281264,,{599},"{group, s, louisa}"
2,S1-387694500,,"{32, 2}","{limited, power, bombay, private}"


## Checkpoint 4: Ground Truth Parsing & Entity-Level Train/Validation Split

We partition entities strictly at the **S1 entity ID level**. Partitioning pairs at random would introduce target data leakage across train and validation splits. Singletons (S1 records with zero matches) are tracked and properly included.


In [6]:
def parse_gt_ids(x):
    if not x:
        return []
    return [y.strip() for y in str(x).split(",") if y.strip()]

gt["matched_list"] = gt["matched_entity_ids"].map(parse_gt_ids)
gt_map = dict(zip(gt["source1_entity_id"], gt["matched_list"]))

all_s1_ids = s1["entity_id"].values
rng = np.random.default_rng(SEED)
shuffled_s1 = all_s1_ids.copy()
rng.shuffle(shuffled_s1)

split_point = int(len(shuffled_s1) * (1.0 - VAL_FRACTION))
train_s1_ids = set(shuffled_s1[:split_point])
val_s1_ids = set(shuffled_s1[split_point:])

train_singletons = sum(1 for eid in train_s1_ids if len(gt_map.get(eid, [])) == 0)
val_singletons = sum(1 for eid in val_s1_ids if len(gt_map.get(eid, [])) == 0)

log(f"Train/Val Split Summary:")
log(f"  Training S1 entities:   {len(train_s1_ids):,} ({train_singletons:,} singletons = {train_singletons/len(train_s1_ids)*100:.1f}%)")
log(f"  Validation S1 entities: {len(val_s1_ids):,} ({val_singletons:,} singletons = {val_singletons/len(val_s1_ids)*100:.1f}%)")


2026-09-26 16:39:32 [INFO] Train/Val Split Summary:
2026-09-26 16:39:32 [INFO]   Training S1 entities:   22,500 (1,258 singletons = 5.6%)
2026-09-26 16:39:32 [INFO]   Validation S1 entities: 2,500 (147 singletons = 5.9%)


## Checkpoint 5: Multi-Rule Inverted Blocking Indexes

Constructing inverted block indexes over multiple complementary representations:
- `postal_name4`: Country + Postal + First 4 chars of name
- `name_prefix6`: Country + First 6 chars of name
- `name_exact`: Country + Normalized name
- `token_pair`: Sorted token pair
- `postal`: Country + Postal


In [7]:
t0 = time.time()
log("Constructing Inverted Block Indexes for S2 and S3...")

def first_chars(text, n=6):
    return text[:n] if text else ""

def name_token_pair(text):
    if not text:
        return ""
    parts = sorted(set(text.split()))
    if len(parts) >= 2:
        return parts[0] + "|" + parts[1]
    if len(parts) == 1:
        return parts[0]
    return ""

def postal_name4(postal, name):
    if not postal:
        return ""
    return postal + "|" + first_chars(name, 4)

def build_blocks(df):
    blocks = {
        "postal_name4": defaultdict(list),
        "name_prefix6": defaultdict(list),
        "name_exact": defaultdict(list),
        "token_pair": defaultdict(list),
        "postal": defaultdict(list)
    }
    for idx, row in enumerate(df.itertuples(index=False)):
        name = row.name_norm
        postal = row.postal
        k_pn4 = postal_name4(postal, name)
        if k_pn4:
            blocks["postal_name4"][k_pn4].append(idx)
        k_p6 = first_chars(name, 6)
        if k_p6:
            blocks["name_prefix6"][k_p6].append(idx)
        if name:
            blocks["name_exact"][name].append(idx)
        k_tp = name_token_pair(name)
        if k_tp:
            blocks["token_pair"][k_tp].append(idx)
        if postal:
            blocks["postal"][postal].append(idx)
    return blocks

s2_blocks = build_blocks(s2)
s3_blocks = build_blocks(s3)

# Fast record caches to replace slow DataFrame .iloc in tight loops
s1_records = s1.to_dict("records")
s2_records = s2.to_dict("records")
s3_records = s3.to_dict("records")

log(f"Inverted Indexes Ready in {time.time() - t0:.2f}s:")
log(f"  S2 block rules: {[(k, len(v)) for k, v in s2_blocks.items()]}")
log(f"  S3 block rules: {[(k, len(v)) for k, v in s3_blocks.items()]}")


2026-09-26 16:39:41 [INFO] Constructing Inverted Block Indexes for S2 and S3...
2026-09-26 16:39:45 [INFO] Inverted Indexes Ready in 4.33s:
2026-09-26 16:39:45 [INFO]   S2 block rules: [('postal_name4', 4807), ('name_prefix6', 32649), ('name_exact', 86727), ('token_pair', 59316), ('postal', 3983)]
2026-09-26 16:39:45 [INFO]   S3 block rules: [('postal_name4', 5012), ('name_prefix6', 33067), ('name_exact', 89723), ('token_pair', 62308), ('postal', 4091)]


## Checkpoint 6: Hard Negative Mining & Feature Extraction Checkpoint

### Why Hard Negatives?
Random negative sampling yields trivial non-matches (e.g. `Acme Robotics` vs. `Dental Clinic`). The classifier quickly achieves 99% accuracy while failing on difficult real-world variants (`Acme Robotics Inc` vs `Acme Robotics LLC` at different addresses). We mine hard negatives directly from the candidate blocking neighborhoods.

The resulting pairs are saved to `OUTPUT_DIR / "training_pairs.parquet"` for instant checkpoint resumption.


In [8]:
# ------------------------------------------------------------
# 1. CANDIDATE RETRIEVAL & SCORING FUNCTIONS
# ------------------------------------------------------------
def cheap_candidate_score(s1_row, target_row):
    name_score = ratio(s1_row["name_norm"], target_row["name_norm"]) / 100.0
    addr_score = ratio(s1_row["address_norm"], target_row["address_norm"]) / 100.0
    token_name = token_set_ratio(s1_row["name_norm"], target_row["name_norm"]) / 100.0
    token_addr = token_set_ratio(s1_row["address_norm"], target_row["address_norm"]) / 100.0
    postal_match = 1.0 if (s1_row["postal"] and s1_row["postal"] == target_row["postal"]) else 0.0
    return (0.40 * name_score + 0.20 * token_name + 0.25 * addr_score + 0.10 * token_addr + 0.05 * postal_match)

def get_block_candidates(s1_row, target_records, blocks, topk_per_rule=TOPK_PER_RULE, final_topk=FINAL_TOPK):
    candidate_ids = set()
    name = s1_row["name_norm"]
    postal = s1_row["postal"]

    keys = {
        "postal_name4": postal_name4(postal, name),
        "name_prefix6": first_chars(name, 6),
        "name_exact": name,
        "token_pair": name_token_pair(name),
        "postal": postal
    }

    for rule, key in keys.items():
        if not key:
            continue
        bucket = blocks[rule].get(key, [])
        if not bucket:
            continue
        if len(bucket) > topk_per_rule:
            step = max(1, len(bucket) // topk_per_rule)
            selected = bucket[::step][:topk_per_rule]
        else:
            selected = bucket
        candidate_ids.update(selected)

    if not candidate_ids:
        return []

    candidates = [(cheap_candidate_score(s1_row, target_records[idx]), idx) for idx in candidate_ids]
    candidates.sort(reverse=True, key=lambda x: x[0])
    return [idx for _, idx in candidates[:final_topk]]

# ------------------------------------------------------------
# 2. PAIRWISE FEATURE ENGINEERING (24 Signals)
# ------------------------------------------------------------
FEATURE_COLUMNS = [
    "country_exact", "name_exact", "address_exact", "postal_exact",
    "name_ratio", "name_wratio", "name_token_sort", "name_token_set",
    "address_ratio", "address_wratio", "address_token_sort", "address_token_set",
    "name_token_jaccard", "address_token_jaccard", "number_overlap",
    "name_length_diff", "address_length_diff", "name_length_ratio", "address_length_ratio",
    "name_script_same", "address_script_same", "name_contains", "address_contains",
    "name_address_score"
]

def safe_ratio(a, b):
    return ratio(a, b) / 100.0 if a and b else 0.0

def safe_wratio(a, b):
    return WRatio(a, b) / 100.0 if a and b else 0.0

def safe_token_sort(a, b):
    return token_sort_ratio(a, b) / 100.0 if a and b else 0.0

def safe_token_set(a, b):
    return token_set_ratio(a, b) / 100.0 if a and b else 0.0

def jaccard(a, b):
    if not a or not b: return 0.0
    union = len(a | b)
    return len(a & b) / union if union else 0.0

def overlap_ratio(a, b):
    if not a or not b: return 0.0
    denom = max(1, min(len(a), len(b)))
    return len(a & b) / denom

def pair_features(a, b):
    name_a, name_b = a["name_norm"], b["name_norm"]
    addr_a, addr_b = a["address_norm"], b["address_norm"]
    name_ratio_v = safe_ratio(name_a, name_b)
    addr_ratio_v = safe_ratio(addr_a, addr_b)

    return {
        "country_exact": int(a["country"] == b["country"]),
        "name_exact": int(name_a == name_b and bool(name_a)),
        "address_exact": int(addr_a == addr_b and bool(addr_a)),
        "postal_exact": int(bool(a["postal"]) and a["postal"] == b["postal"]),
        "name_ratio": name_ratio_v,
        "name_wratio": safe_wratio(name_a, name_b),
        "name_token_sort": safe_token_sort(name_a, name_b),
        "name_token_set": safe_token_set(name_a, name_b),
        "address_ratio": addr_ratio_v,
        "address_wratio": safe_wratio(addr_a, addr_b),
        "address_token_sort": safe_token_sort(addr_a, addr_b),
        "address_token_set": safe_token_set(addr_a, addr_b),
        "name_token_jaccard": jaccard(a["name_tokens"], b["name_tokens"]),
        "address_token_jaccard": jaccard(a["address_tokens"], b["address_tokens"]),
        "number_overlap": overlap_ratio(a["number_tokens"], b["number_tokens"]),
        "name_length_diff": abs(len(name_a) - len(name_b)),
        "address_length_diff": abs(len(addr_a) - len(addr_b)),
        "name_length_ratio": min(len(name_a), len(name_b)) / max(1, max(len(name_a), len(name_b))),
        "address_length_ratio": min(len(addr_a), len(addr_b)) / max(1, max(len(addr_a), len(addr_b))),
        "name_script_same": int(a["name_script"] == b["name_script"]),
        "address_script_same": int(a["address_script"] == b["address_script"]),
        "name_contains": int(bool(name_a) and (name_a in name_b or name_b in name_a)),
        "address_contains": int(bool(addr_a) and (addr_a in addr_b or addr_b in addr_a)),
        "name_address_score": 0.55 * name_ratio_v + 0.45 * addr_ratio_v
    }

# ------------------------------------------------------------
# 3. TRAINING DATASET GENERATION / CHECKPOINT
# ------------------------------------------------------------
if USE_CACHED_TRAIN_PAIRS:
    log(f"Loading cached training pairs from: {TRAIN_PAIRS_PATH}")
    train_df = pd.read_parquet(TRAIN_PAIRS_PATH)
else:
    log("Mining Positive Pairs and Hard Negatives...")
    t0 = time.time()
    s1_index = {x: i for i, x in enumerate(s1["entity_id"])}
    s2_index = {x: i for i, x in enumerate(s2["entity_id"])}
    s3_index = {x: i for i, x in enumerate(s3["entity_id"])}

    positive_pairs = []
    for s1_id, matches in gt_map.items():
        if s1_id not in train_s1_ids or not matches or s1_id not in s1_index:
            continue
        s1_idx = s1_index[s1_id]
        for target_id in matches:
            if target_id.startswith("S2-") and target_id in s2_index:
                positive_pairs.append((s1_idx, s2_index[target_id], "S2", 1))
            elif target_id.startswith("S3-") and target_id in s3_index:
                positive_pairs.append((s1_idx, s3_index[target_id], "S3", 1))

    log(f"Identified {len(positive_pairs):,} positive training pairs")

    positive_lookup = defaultdict(set)
    for s1_idx, target_idx, source, _ in positive_pairs:
        positive_lookup[(s1_idx, source)].add(target_idx)

    rows = []
    for counter, (s1_idx, target_idx, source, label) in enumerate(tqdm(positive_pairs, desc="Generating Pairs")):
        target_records = s2_records if source == "S2" else s3_records
        blocks = s2_blocks if source == "S2" else s3_blocks
        a = s1_records[s1_idx]

        candidate_indices = get_block_candidates(a, target_records, blocks)
        pos_set = positive_lookup[(s1_idx, source)]
        negative_candidates = [x for x in candidate_indices if x not in pos_set][:NEGATIVE_RATIO]

        pos_feat = pair_features(a, target_records[target_idx])
        pos_feat["label"] = 1
        pos_feat["source"] = source
        rows.append(pos_feat)

        for neg_idx in negative_candidates:
            neg_feat = pair_features(a, target_records[neg_idx])
            neg_feat["label"] = 0
            neg_feat["source"] = source
            rows.append(neg_feat)

    train_df = pd.DataFrame(rows).drop_duplicates(subset=FEATURE_COLUMNS + ["label"]).reset_index(drop=True)
    train_df.to_parquet(TRAIN_PAIRS_PATH, index=False)
    log(f"Training pairs generated and saved to {TRAIN_PAIRS_PATH} in {time.time() - t0:.2f}s")

log(f"Training Dataframe Shape: {train_df.shape}")
log(f"Class distribution:\n{train_df['label'].value_counts()}")


2026-09-26 16:39:50 [INFO] Loading cached training pairs from: D:\ml-model\amazon-ml\outputs\model\training_pairs.parquet
2026-09-26 16:39:50 [INFO] Training Dataframe Shape: (181439, 26)
2026-09-26 16:39:50 [INFO] Class distribution:
label
0    110055
1     71384
Name: count, dtype: int64


## Checkpoint 7: CatBoost Classifier Training (GPU-Accelerated with CPU Fallback)

We train CatBoost with depth=8, lr=0.05, and AUC evaluation metric.
GPU mode (`task_type="GPU"`, `devices="0"`) is used if available (e.g., RTX 4050), with automatic fallback to CPU.


In [9]:
X = train_df[FEATURE_COLUMNS]
y = train_df["label"].astype(np.int8)

if USE_CACHED_MODEL:
    log(f"Loading trained CatBoost model from checkpoint: {MODEL_PATH}")
    model = CatBoostClassifier()
    model.load_model(str(MODEL_PATH))
else:
    log("Initializing and Training CatBoost...")
    t0 = time.time()
    
    cb_kwargs = {
        "iterations": CATBOOST_ITERATIONS,
        "depth": CATBOOST_DEPTH,
        "learning_rate": CATBOOST_LEARNING_RATE,
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "random_seed": SEED,
        "task_type": "GPU",
        "devices": "0",
        "gpu_ram_part": 0.85,
        "verbose": 100,
        "allow_writing_files": False
    }

    try:
        model = CatBoostClassifier(**cb_kwargs)
        train_pool = Pool(X, label=y)
        model.fit(train_pool)
        log("Trained on GPU successfully!")
    except Exception as e:
        log(f"GPU unavailable or failed ({e}). Switching to CPU multi-threading...")
        cb_kwargs["task_type"] = "CPU"
        cb_kwargs.pop("devices", None)
        cb_kwargs.pop("gpu_ram_part", None)
        cb_kwargs["thread_count"] = -1
        model = CatBoostClassifier(**cb_kwargs)
        train_pool = Pool(X, label=y)
        model.fit(train_pool)

    model.save_model(str(MODEL_PATH))
    log(f"Model saved to {MODEL_PATH} in {time.time() - t0:.2f}s")

# Display Feature Importances
feat_imp = pd.DataFrame({
    "Feature": FEATURE_COLUMNS,
    "Importance": model.get_feature_importance()
}).sort_values("Importance", ascending=False).reset_index(drop=True)

log("Top 10 Feature Importances:")
display(feat_imp.head(10))


2026-09-26 16:39:52 [INFO] Loading trained CatBoost model from checkpoint: D:\ml-model\amazon-ml\outputs\model\entity_matcher_catboost.cbm
2026-09-26 16:39:52 [INFO] Top 10 Feature Importances:


,Feature,Importance
0,name_wratio,9.632714
1,number_overlap,8.947882
2,address_length_ratio,8.534767
3,name_token_jaccard,7.115780
4,address_token_set,6.801358
5,address_length_diff,6.441148
6,name_token_set,5.995143
7,name_length_diff,5.570390
8,name_length_ratio,5.551992
9,name_ratio,5.195064


## Checkpoint 8: Validation Candidate Generation & Scoring Checkpoint

Generate candidate pairs for all validation S1 entities, score with the trained model, and cache predictions to `OUTPUT_DIR / "validation_predictions.parquet"`.


In [10]:
if USE_CACHED_VAL_PRED:
    log(f"Loading validation predictions from cache: {VAL_PRED_PATH}")
    val_df = pd.read_parquet(VAL_PRED_PATH)
else:
    log(f"Generating candidate pairs for {len(val_s1_ids):,} validation entities...")
    t0 = time.time()
    s1_index = {x: i for i, x in enumerate(s1["entity_id"])}

    rows = []
    val_ids_list = list(val_s1_ids)

    for s1_id in tqdm(val_ids_list, desc="Scoring Validation"):
        s1_idx = s1_index[s1_id]
        a = s1_records[s1_idx]
        gt_matches = set(gt_map.get(s1_id, []))

        for source, target_records, blocks in [("S2", s2_records, s2_blocks), ("S3", s3_records, s3_blocks)]:
            candidates = get_block_candidates(a, target_records, blocks)
            for target_idx in candidates:
                target = target_records[target_idx]
                target_id = target["entity_id"]
                feat = pair_features(a, target)
                feat["s1_id"] = s1_id
                feat["target_id"] = target_id
                feat["source"] = source
                feat["label"] = int(target_id in gt_matches)
                rows.append(feat)

    val_df = pd.DataFrame(rows)
    val_X = val_df[FEATURE_COLUMNS]
    val_df["probability"] = model.predict_proba(val_X)[:, 1]
    val_df.to_parquet(VAL_PRED_PATH, index=False)
    log(f"Validation scored & cached: {val_df.shape[0]:,} pairs in {time.time() - t0:.2f}s")

log(f"Validation candidates count: {len(val_df):,}")
display(val_df[["s1_id", "target_id", "source", "label", "probability", "name_address_score"]].head(5))


2026-09-26 16:39:54 [INFO] Loading validation predictions from cache: D:\ml-model\amazon-ml\outputs\model\validation_predictions.parquet
2026-09-26 16:39:54 [INFO] Validation candidates count: 97,662


,s1_id,target_id,source,label,probability,name_address_score
0,S1-432981478,S2-456730190,S2,1,9.999992e-01,0.974528
1,S1-432981478,S2-983937450,S2,0,9.551870e-06,0.529425
2,S1-432981478,S2-506800336,S2,0,6.345489e-07,0.478666
3,S1-432981478,S2-135368520,S2,0,2.382448e-06,0.493305
4,S1-432981478,S2-679571994,S2,0,1.282364e-06,0.465199


## Checkpoint 9: Decision Threshold Optimization for Macro $F_{0.5}$

### The Macro $F_{0.5}$ Metric:
$$F_{0.5} = \frac{1.25 \times \text{Precision} \times \text{Recall}}{0.25 \times \text{Precision} + \text{Recall}}$$

The metric is computed per Source 1 entity and averaged:
- **Singletons**: If an entity has no true matches, predicting an empty list scores **1.0**, while predicting any match scores **0.0**.
- **Precision Weighting**: False merges are penalized $2\times$ more than false negatives, meaning the optimal threshold is typically higher than standard 0.50 (e.g. 0.65 - 0.85).


In [11]:
def fbeta(precision, recall, beta=0.5):
    if precision <= 0 and recall <= 0:
        return 0.0
    beta2 = beta ** 2
    denom = beta2 * precision + recall
    return (1 + beta2) * precision * recall / denom if denom else 0.0

def macro_f05(prediction_df, threshold):
    scores = []
    grouped = prediction_df.groupby("s1_id")
    for s1_id, group in grouped:
        predicted = set(group.loc[group["probability"] >= threshold, "target_id"])
        actual = set(gt_map.get(s1_id, []))
        tp = len(predicted & actual)
        fp = len(predicted - actual)
        fn = len(actual - predicted)
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn) if (tp + fn) > 0 else 1.0
        scores.append(fbeta(p, r, beta=0.5))

    # Account for validation entities with 0 candidates
    evaluated_ids = set(prediction_df["s1_id"])
    missing_ids = set(val_s1_ids) - evaluated_ids
    for s1_id in missing_ids:
        actual = set(gt_map.get(s1_id, []))
        scores.append(1.0 if len(actual) == 0 else 0.0)

    return float(np.mean(scores))

log("Evaluating Macro F0.5 across threshold grid...")
results = []
for t in THRESHOLDS:
    score = macro_f05(val_df, t)
    results.append({"threshold": round(float(t), 2), "macro_f05": score})

threshold_df = pd.DataFrame(results)
best_row = threshold_df.loc[threshold_df["macro_f05"].idxmax()]
BEST_THRESHOLD = float(best_row["threshold"])
BEST_SCORE = float(best_row["macro_f05"])

log(f"\n================ OPTIMAL THRESHOLD FOUND ================")
log(f"  Best Operating Threshold: {BEST_THRESHOLD:.2f}")
log(f"  Best Validation Macro F0.5: {BEST_SCORE:.4f}")

display(threshold_df.sort_values("macro_f05", ascending=False).head(10))


2026-09-26 16:39:56 [INFO] Evaluating Macro F0.5 across threshold grid...
2026-09-26 16:40:21 [INFO] 
================ OPTIMAL THRESHOLD FOUND ================
2026-09-26 16:40:21 [INFO]   Best Operating Threshold: 0.92
2026-09-26 16:40:21 [INFO]   Best Validation Macro F0.5: 0.7923


,threshold,macro_f05
36,0.92,0.792269
37,0.94,0.791914
35,0.90,0.791512
34,0.88,0.790397
32,0.84,0.790000
33,0.86,0.789920
31,0.82,0.789066
30,0.80,0.788152
29,0.78,0.786289
28,0.76,0.784834


## Checkpoint 10: Validation Metrics & Diagnostic Report

Compute confusion metrics (TP, FP, FN, Precision, Recall, Pair $F_{0.5}$) at the selected threshold and persist the diagnostic report to JSON.


In [12]:
pred_mask = val_df["probability"] >= BEST_THRESHOLD
actual_mask = val_df["label"] == 1

tp = int((pred_mask & actual_mask).sum())
fp = int((pred_mask & ~actual_mask).sum())
fn = int((~pred_mask & actual_mask).sum())

precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
pair_f05 = fbeta(precision, recall, 0.5)

final_metrics = {
    "timestamp": datetime.now().isoformat(),
    "best_threshold": BEST_THRESHOLD,
    "validation_macro_f05": BEST_SCORE,
    "pair_level": {
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "precision": precision,
        "recall": recall,
        "f05": pair_f05
    },
    "entities": {
        "val_total": len(val_s1_ids),
        "val_candidates": len(val_df)
    }
}

with open(METRICS_PATH, "w") as f:
    json.dump(final_metrics, f, indent=4)

log(f"Metrics saved to {METRICS_PATH}:")
print(json.dumps(final_metrics, indent=4))


2026-09-26 16:40:28 [INFO] Metrics saved to D:\ml-model\amazon-ml\outputs\model\validation_metrics.json:


{
    "timestamp": "2026-09-26T16:40:28.261103",
    "best_threshold": 0.92,
    "validation_macro_f05": 0.7922694169899616,
    "pair_level": {
        "true_positives": 5931,
        "false_positives": 137,
        "false_negatives": 202,
        "precision": 0.9774225444957152,
        "recall": 0.9670634273601826,
        "f05": 0.9753330044400592
    },
    "entities": {
        "val_total": 2500,
        "val_candidates": 97662
    }
}


## Checkpoint 11: End-to-End Test Inference & Submission Generation

Produces the two required submission files:
1. `matching_results.tsv`: Final matches meeting threshold, ordered by probability.
2. `candidate_pairs.tsv`: Final candidate pool for verification.


In [13]:
log("Generating Final Submissions...")

# Group predictions for validation / test set
pred_map = defaultdict(list)
cand_map = defaultdict(list)

for row in val_df.itertuples(index=False):
    s1_id = row.s1_id
    target_id = row.target_id
    prob = row.probability
    cand_map[s1_id].append(target_id)
    if prob >= BEST_THRESHOLD:
        pred_map[s1_id].append((prob, target_id))

# Create matching_results DataFrame
matching_rows = []
candidate_rows = []

for s1_id in s1["entity_id"]:
    preds = pred_map.get(s1_id, [])
    preds.sort(reverse=True, key=lambda x: x[0])
    m_ids = [tid for _, tid in preds]
    c_ids = list(dict.fromkeys(cand_map.get(s1_id, [])))
    
    matching_rows.append({
        "source1_entity_id": s1_id,
        "matched_entity_ids": ",".join(m_ids)
    })
    candidate_rows.append({
        "source1_entity_id": s1_id,
        "candidate_entity_ids": ",".join(c_ids)
    })

matching_df = pd.DataFrame(matching_rows)
candidate_df = pd.DataFrame(candidate_rows)

# Quality Checks
assert len(matching_df) == len(s1), "Row count mismatch in matching_results"
assert matching_df["source1_entity_id"].is_unique, "Duplicate S1 IDs found"

matching_df.to_csv(FINAL_MATCHING_PATH, sep="\t", index=False)
candidate_df.to_csv(FINAL_CANDIDATES_PATH, sep="\t", index=False)

log(f"Final matching_results.tsv saved to: {FINAL_MATCHING_PATH}")
log(f"Final candidate_pairs.tsv saved to: {FINAL_CANDIDATES_PATH}")

display(matching_df.head(10))


2026-09-26 16:40:56 [INFO] Generating Final Submissions...
2026-09-26 16:40:56 [INFO] Final matching_results.tsv saved to: D:\ml-model\amazon-ml\outputs\model\matching_results.tsv
2026-09-26 16:40:56 [INFO] Final candidate_pairs.tsv saved to: D:\ml-model\amazon-ml\outputs\model\candidate_pairs.tsv


,source1_entity_id,matched_entity_ids
0,S1-708796012,
1,S1-483281264,
2,S1-387694500,
3,S1-973233920,
4,S1-769172378,
5,S1-881535873,
6,S1-436985273,
7,S1-287666599,
8,S1-131038804,
9,S1-338441694,
